In [ ]:
import math
import os
import csv
import sys
import getpass
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import GPT2TokenizerFast


import string
import random

DROPOUT = 0.1

In [ ]:
ds = load_dataset("cimec/lambada")

print(len(ds["train"]["text"]))
print(len(ds["validation"]["text"]))
print(len(ds["test"]))

In [ ]:
def get_last_word(text: str):
    words = text.split()
    last = words[-1] if words else ""
    others = " ".join(words[:-1])+" "
    return (others), (last)

In [ ]:
class CharTokenizer:
    def __init__(self, text: str):
        self.special = ["<pad>", "<bos>", "<eos>", "<unk>"]
        chars = sorted(set(text)) + self.special

        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for c, i in self.stoi.items()}
        self.vocab_size = len(chars)

        self.pad_token_id = self.stoi["<pad>"]
        self.eos_token_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text: str, add_eos=True):
        ids = [self.stoi.get(c, self.unk_id) for c in text]
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)

In [ ]:
#stable_vocab = string.printable 
#tokenizer = CharTokenizer(stable_vocab)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def pad(seq: torch.Tensor, target_len: int, pad_token_id):
    seq = seq[:target_len]
    if len(seq) < target_len:
        seq = torch.cat([
            seq,
            torch.full((target_len - len(seq),), pad_token_id, dtype=torch.long)
        ])
    return seq

In [ ]:
def build_packed_stream(tokenized_texts, eos_token_id):
    lengths = sum(len(seq) + 1 for seq in tokenized_texts)

    arr = torch.empty(lengths, dtype=torch.long)

    i = 0
    for seq in tokenized_texts:
        n = len(seq)
        arr[i:i+n] = torch.tensor(seq, dtype=torch.long)
        i += n

        arr[i] = eos_token_id
        i += 1

    return arr

In [ ]:
class PackedTextDataset(Dataset):
    def __init__(self, stream, block_size):
        self.stream = stream
        self.block_size = block_size

        self.n_blocks = (len(stream) - 1) // block_size

    def __len__(self):
        return self.n_blocks

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size + 1

        chunk = self.stream[start:end]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y

In [ ]:
class WordEmbedding(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, context_len: int):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_len, embed_dim)
        self.context_len = context_len

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        tok = self.token_emb(idx)

        pos = torch.arange(T, device=idx.device)
        pos = self.pos_emb(pos)[None, :, :]

        return tok + pos

In [ ]:
#correct but old and doesnt use the hardware acceleration that F.scaled_dot_product_attention uses
class CustomMaskedMultiHeadSelfAttn(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, context_len: int):
        super().__init__()
        
        self.embed_dim   = embed_dim
        self.num_heads   = num_heads

        self.head_dim    = embed_dim // num_heads

        self.q_proj    = nn.Linear(embed_dim, embed_dim)
        self.k_proj    = nn.Linear(embed_dim, embed_dim)
        self.v_proj    = nn.Linear(embed_dim, embed_dim)
        self.out_proj    = nn.Linear(embed_dim, embed_dim)

        mask = torch.tril(torch.ones(context_len, context_len))

        self.register_buffer("mask", mask.view(1, 1, context_len, context_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

        #attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(
            torch.softmax(attn_scores, dim=-1)
        )

        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

In [ ]:
#3-4x faster than old doing very similar calculation but utilizing hardware acceleration better 
#especially with torch.amp.autocast(device_type=DEVICE, dtype=torch.bfloat16):
class MaskedMultiHeadSelfAttn(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, context_len: int):
        super().__init__()
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=None, 
            dropout_p=0.0, 
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, context_len: int):
        super().__init__()

        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = MaskedMultiHeadSelfAttn(
            embed_dim=embed_dim,
            num_heads=num_heads,
            context_len=context_len
        )

        self.ln2  = nn.LayerNorm(embed_dim)

        #self.ff = nn.Sequential(
        #    nn.Linear(embed_dim, ff_dim),
        #    nn.ReLU(),
        #    nn.Linear(ff_dim, embed_dim)
        #)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
CONTEXT_LEN  = 256     # sequence length (tokens)
EMBED_DIM    = 256      # model width
NUM_HEADS    = 4        # attention heads  (must divide EMBED_DIM)
NUM_LAYERS   = 4        # transformer blocks
FF_DIM       = 512      # feed-forward hidden size

BATCH_SIZE   = 4
EPOCHS       = 1000
LR           = 3e-4
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

WARMUP_EPOCHS = max(10, EPOCHS // 50)

In [ ]:
class RNNLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2, context_len=256):
        super().__init__()

        self.embedding = WordEmbedding(vocab_size, embed_dim, context_len)

        self.token_emb = self.embedding.token_emb

        self.lstm = nn.LSTM(
            input_size=self.token_emb.embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.ln = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, idx, targets=None):
        x = self.token_emb(idx)

        x, _ = self.lstm(x)
        x = self.ln(x)

        logits = self.fc(x)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.view(B*T, V),
                targets.view(B*T),
                ignore_index=tokenizer.pad_token_id
            )

        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx: torch.Tensor, eos_token_id, max_new_tokens: int=128,temperature: float=1.0,top_k: int=None):
        hidden = None

        if idx.size(1) > 1:
            x = self.token_emb(idx[:, :-1])
            _, hidden = self.lstm(x)

        current = idx[:, -1:]

        finished = torch.zeros(
            idx.size(0),
            dtype=torch.bool,
            device=idx.device
        )

        for _ in range(max_new_tokens):

            x = self.token_emb(current)

            out, hidden = self.lstm(x, hidden)

            logits = self.fc(self.ln(out[:, -1]))
            logits = logits / temperature

            if top_k is not None:
                values, _ = torch.topk(logits, top_k)

                logits = torch.where(
                    logits < values[:, -1].unsqueeze(-1),
                    torch.tensor(float("-inf"), device=logits.device),
                    logits
                )

            probs = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat([idx, next_token], dim=1)

            finished |= (next_token.squeeze(-1) == eos_token_id)

            if finished.all():
                break

            current = next_token

        return idx

In [ ]:
class TransformerSimple(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = EMBED_DIM, num_heads: int = NUM_HEADS, num_layers: int = NUM_LAYERS, ff_dim: int = FF_DIM, context_len: int = CONTEXT_LEN):
        super().__init__()

        self.embedding = WordEmbedding(vocab_size, embed_dim, context_len)
        
        self.blocks    = nn.Sequential(*[
            AttentionBlock(embed_dim, num_heads, ff_dim, context_len)
            for _ in range(num_layers)
        ])
        
        self.ln_f      = nn.LayerNorm(embed_dim)
        self.head      = nn.Linear(embed_dim, vocab_size, bias=False)

        self.emb_dropout = nn.Dropout(DROPOUT)

        self.head.weight = self.embedding.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor=None):
        #x = self.embedding(idx)
        x = self.emb_dropout(self.embedding(idx))
    
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            logits_flat = logits.reshape(B * T, V)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=tokenizer.pad_token_id)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, eos_token_id, max_new_tokens: int=128, temperature: float = 1.0, top_k: int=None) -> torch.Tensor:
        ctx = self.embedding.context_len
        B = idx.size(0)

        finished = torch.zeros(B, dtype=torch.bool, device=idx.device)

        for _ in range(max_new_tokens):

            if finished.all():
                break

            idx_cond = idx[:, -ctx:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                thresh = v[:, -1].unsqueeze(-1)
                logits = torch.where(
                    logits < thresh,
                    torch.tensor(float("-inf"), device=logits.device),
                    logits
                )

            probs = torch.softmax(logits, dim=-1)
            next_t = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_t], dim=1)

            finished |= (next_t.squeeze(-1) == eos_token_id)

        return idx

In [ ]:
def train(model: TransformerSimple, loader: DataLoader,
          optimizer: torch.optim.Optimizer  ) -> float:
    model.train()
    total_loss = 0.0
    i = 1
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        #_, loss = model(x, y)
        with torch.amp.autocast(device_type=DEVICE, dtype=torch.bfloat16):
            _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        if i%(len(loader)//1000) == 0:
            return total_loss / i
            print(f"Batch Loss: {loss.item():.4f}, {round(i/len(loader), 2):.2f}")
        i += 1
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model: TransformerSimple, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        #_, loss = model(x, y)
        with torch.amp.autocast(device_type=DEVICE, dtype=torch.bfloat16):
            _, loss = model(x, y)

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate_next_token_accuracy(model: TransformerSimple, loader: DataLoader):
    model.eval()

    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits, _ = model(x)

        preds = logits.argmax(dim=-1)

        correct += (preds == y).sum().item()
        total += y.numel()

    return correct / total

@torch.no_grad()
def evaluate_final_word_accuracy(model: nn.Module, test_tuples, tokenizer, batch_size=128):
    model.eval()
    
    correct = 0
    total = 0
    
    valid_tuples = [(o, l) for o, l in test_tuples if l.strip()]
    
    for i in range(0, len(valid_tuples), batch_size):
        batch = valid_tuples[i : i + batch_size]
        
        batch_contexts = []
        context_lens = []
        target_words = []
        max_target_chars = 0
        
        for others, last in batch:
            c_ids = tokenizer.encode(others, add_special_tokens=False)
            
            if len(c_ids) > (CONTEXT_LEN - 16):
                c_ids = c_ids[-(CONTEXT_LEN - 16):]
                
            batch_contexts.append(torch.tensor(c_ids, dtype=torch.long))
            context_lens.append(len(c_ids))
            target_words.append(last.strip())
            max_target_chars = max(max_target_chars, len(last))
            
        if not batch_contexts:
            continue
            
        max_context_len = max(context_lens)
        padded_x = torch.stack([
            pad(seq, max_context_len, tokenizer.pad_token_id) for seq in batch_contexts
        ]).to(DEVICE)
        
        max_new_tokens = max(3, (max_target_chars // 2) + 4)
        
        #generated_ids = model.generate(
        #    padded_x, 
        #    eos_token_id=tokenizer.eos_token_id, 
        #    max_new_tokens=max_new_tokens, 
        #    temperature=1.0, 
        #    top_k=None
        #)
        with torch.amp.autocast(device_type=DEVICE, dtype=torch.bfloat16):
            generated_ids = model.generate(
                padded_x, 
                eos_token_id=tokenizer.eos_token_id, 
                max_new_tokens=max_new_tokens, 
                temperature=1.0, 
                top_k=None
            )
        
        for b_idx in range(len(batch)):
            start_offset = context_lens[b_idx]
            padding_offset = max_context_len - start_offset
            
            actual_gen_start = max_context_len
            new_tokens = generated_ids[b_idx, actual_gen_start:].tolist()
            
            predicted_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            predicted_word = predicted_text.split()[0] if predicted_text.split() else ""
            if predicted_word == target_words[b_idx]:
                correct += 1
            total += 1
            
    return correct / total if total > 0 else 0.0

In [ ]:
data_set = load_dataset("cimec/lambada")

train_data_set = data_set["train"]

val_data_set = data_set["validation"]

test_data_set = data_set["test"]["text"]

test_start_finish = [get_last_word(item) for item in test_data_set]

print(f"Vocabulary size : {tokenizer.vocab_size}")

#train_tok = [torch.tensor(tokenizer.encode(item["text"]), dtype=torch.long) for item in train_data_set]
#val_tok = [torch.tensor(tokenizer.encode(item["text"]), dtype=torch.long) for item in val_data_set]
#train_tok = [torch.tensor(tokenizer.encode(item["text"], add_special_tokens=False), dtype=torch.long) for item in train_data_set]
#val_tok = [torch.tensor(tokenizer.encode(item["text"], add_special_tokens=False), dtype=torch.long) for item in val_data_set]

#print("Encoded")

train_stream = None
val_stream = None

#train_stream = build_packed_stream(train_tok, tokenizer.eos_token_id)
#val_stream = build_packed_stream(val_tok, tokenizer.eos_token_id)

if os.path.exists("train_stream.pt"):
    train_stream = torch.load("train_stream.pt")
    val_stream = torch.load("val_stream.pt")
    print("Loaded Encoded")
else:
    train_tok = [torch.tensor(tokenizer.encode(item["text"], add_special_tokens=False), dtype=torch.long) for item in train_data_set]
    val_tok = [torch.tensor(tokenizer.encode(item["text"], add_special_tokens=False), dtype=torch.long) for item in val_data_set]

    train_stream = build_packed_stream(train_tok, tokenizer.eos_token_id)
    val_stream = build_packed_stream(val_tok, tokenizer.eos_token_id)

    torch.save(train_stream, "train_stream.pt")
    torch.save(val_stream, "val_stream.pt")
    print("Created Encoded")

train_set  = PackedTextDataset(train_stream, CONTEXT_LEN)
val_set    = PackedTextDataset(val_stream, CONTEXT_LEN)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

print(f"Train samples: {len(train_set):,}  |  Val samples: {len(val_set):,}")

In [ ]:
LSTMSmall = RNNLSTM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    context_len = CONTEXT_LEN,
)

LSTMMedium = RNNLSTM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=256,
    hidden_dim=512,
    num_layers=3,
    context_len = CONTEXT_LEN,
)

LSTMLarge = RNNLSTM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=512,
    hidden_dim=1024,
    num_layers=4,
    context_len = CONTEXT_LEN,
)

transformerTiny = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 128,
    num_heads   = 2,
    num_layers  = 2,
    ff_dim      = 256,
    context_len = CONTEXT_LEN,
)

transformerSmall = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 256,
    num_heads   = 4,
    num_layers  = 6,
    ff_dim      = 1024,
    context_len = CONTEXT_LEN,
)

transformerMedium = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 512,
    num_heads   = 8,
    num_layers  = 8,
    ff_dim      = 2048,
    context_len = CONTEXT_LEN,
)

transformerLarge = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 768,
    num_heads   = 12,
    num_layers  = 12,
    ff_dim      = 3072,
    context_len = CONTEXT_LEN,
)

modelList = [
    {"name": "LSTMSmall", "model": LSTMSmall, "batch_size": 16, "optimizer": None, "scheduler": None}, 
    {"name": "LSTMMedium", "model": LSTMMedium, "batch_size": 8, "optimizer": None, "scheduler": None}, 
    {"name": "LSTMLarge", "model": LSTMLarge, "batch_size": 2, "optimizer": None, "scheduler": None}, 

    {"name": "transformerTiny", "model": transformerTiny, "batch_size": 32, "optimizer": None, "scheduler": None}, 
    {"name": "transformerSmall", "model": transformerSmall, "batch_size": 16, "optimizer": None, "scheduler": None},
    {"name": "transformerMedium", "model": transformerMedium, "batch_size": 2, "optimizer": None, "scheduler": None},
    {"name": "transformerLarge", "model": transformerLarge, "batch_size": 1, "optimizer": None, "scheduler": None},
]

for i in modelList:
    model = i["model"].to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{i["name"]} parameters : {n_params:,}\n")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)


    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=WARMUP_EPOCHS
    )

    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS - WARMUP_EPOCHS,
        eta_min=LR / 10
    )

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[WARMUP_EPOCHS]
    )

    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    #    optimizer, T_max=EPOCHS, eta_min=LR / 10
    #)

    i["optimizer"] = optimizer
    i["scheduler"] = scheduler

In [ ]:
#transformerLarge.load_state_dict(torch.load("transformerLarge.pth"))

In [ ]:
for epoch in range(1, EPOCHS + 1):
    for item in modelList:
        print("Training model: " + item["name"])

        model = item["model"]
        optimizer = item["optimizer"]
        scheduler = item["scheduler"]

        train_loss = train(model, train_loader, optimizer)
        val_loss = evaluate(model, val_loader)

        current_lr = optimizer.param_groups[0]["lr"]

        val_next_token_acc = None
        test_final_word_acc = None

        scheduler.step()

        if epoch % 10 == 0:
            val_next_token_acc = evaluate_next_token_accuracy(model, val_loader)
            test_final_word_acc = evaluate_final_word_accuracy(model, test_start_finish, tokenizer, batch_size=item["batch_size"])

            print(
                f"Epoch {epoch:>3}/{EPOCHS}  "
                f"train={train_loss:.4f}  "
                f"val={val_loss:.4f}  "
                f"ppl={math.exp(val_loss):.1f}  "
                f"LR={current_lr:.2e}  "
                f"next token acc={val_next_token_acc:.4%}  "
                f"final word acc={test_final_word_acc:.4%}"
            )
        else:
            print(
                f"Epoch {epoch:>3}/{EPOCHS}  "
                f"train={train_loss:.4f}  "
                f"val={val_loss:.4f}  "
                f"ppl={math.exp(val_loss):.1f}  "
                f"LR={current_lr:.2e}  "
            )

        filename = f"{item['name']}.csv"
        file_exists = os.path.isfile(filename)

        with open(filename, mode="a", newline="") as f:
            writer = csv.writer(f)

            if not file_exists:
                writer.writerow([
                    "epoch",
                    "train_loss",
                    "val_loss",
                    "val_next_token_acc",
                    "test_final_word_acc"
                ])

            writer.writerow([
                epoch,
                train_loss,
                val_loss,
                val_next_token_acc if val_next_token_acc is not None else "",
                test_final_word_acc if test_final_word_acc is not None else ""
            ])

In [ ]:
for i in modelList:
    torch.save(i["model"].state_dict(), i["name"]+".pth")

In [ ]:
print("\n── Sample generation (temperature=0.8, top_k=40) ──\n")
model = None
for i in modelList:
    if i["name"] == "transformerLarge":
        model = i["model"]
        break
    
model.eval()

seed_text = "the"
#seed_ids  = torch.tensor(tokenizer.encode(seed_text, add_eos=False), dtype=torch.long, device=DEVICE).unsqueeze(0)
seed_ids  = torch.tensor(tokenizer.encode(seed_text, add_special_tokens=False), dtype=torch.long, device=DEVICE).unsqueeze(0)
out_ids   = model.generate(seed_ids, tokenizer.eos_token_id, 60,
                            temperature=1.0, top_k=40)
print(tokenizer.decode(out_ids[0].tolist(), skip_special_tokens=True))